<a href="https://colab.research.google.com/github/pipithtp/Generative-AI/blob/main/Tugas_GEN_AI_Modul_1_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.2 Variables and Data Types

In [48]:
# Core scalar types
model_name: str = "claude-sonnet-4-5"
temperature: float = 0.7
max_tokens: int = 1024
is_streaming: bool = True

# Check types at runtime
print(type(model_name))
print(type(temperature))

response = None
print(response is None)

<class 'str'>
<class 'float'>
True


Strings

In [49]:
user_input = "Explain transformers in simple terms"
system_prompt = "You are a helpful AI tutor."

# f-strings
full_prompt = f"System: {system_prompt}\nUser: {user_input}"
print(full_prompt)

# Common string methods
print(user_input.upper())
print(user_input.split())
print(user_input.replace("simple", "plain"))
print(len(user_input))

System: You are a helpful AI tutor.
User: Explain transformers in simple terms
EXPLAIN TRANSFORMERS IN SIMPLE TERMS
['Explain', 'transformers', 'in', 'simple', 'terms']
Explain transformers in plain terms
36


Numbers

In [50]:
# Integer arithmetic
tokens_used = 450
tokens_limit = 1024

remaining = tokens_limit - tokens_used  # 574

# Float arithmetic
cost_per_token = 0.000003
total_cost = tokens_used * cost_per_token
print(f"Cost: ${total_cost:.6f}")

# Integer division and modulo
batches = tokens_used // 100  # 4
leftover = tokens_used % 100  # 50

# Built-in math
import math
print(math.log2(512))  # 9.0 - useful in information theory
print(math.ceil(3.1))  # 4

Cost: $0.001350
9.0
4


1.3 Control Flow - if / elif / else

In [51]:
def classify_response_length(token_count: int) -> str:
    if token_count < 100:
        return "short"
    elif token_count < 500:
        return "medium"
    elif token_count < 2000:
        return "long"
    else:
        return "very long"

print(classify_response_length(80))    # short
print(classify_response_length(350))   # medium
print(classify_response_length(3000))  # very long

short
medium
very long


for Loops

In [52]:
models = ["gpt-4o", "claude-sonnet-4-5", "gemini-1.5-pro"]

# Basic iteration
for model in models:
    print(f"Checking: {model}")

# With index - use enumerate, not range(len(...))
for i, model in enumerate(models):
    print(f"{i + 1}. {model}")

# Iterate over key-value pairs in a dict
token_limits = {"gpt-4o": 128000, "claude-sonnet-4-5": 200000}
for model, limit in token_limits.items():
    print(f"{model}: {limit:,} tokens")

Checking: gpt-4o
Checking: claude-sonnet-4-5
Checking: gemini-1.5-pro
1. gpt-4o
2. claude-sonnet-4-5
3. gemini-1.5-pro
gpt-4o: 128,000 tokens
claude-sonnet-4-5: 200,000 tokens


while Loops

In [53]:
import time

MAX_RETRIES = 3
attempt = 0

while attempt < MAX_RETRIES:
    attempt += 1
    print(f"Attempt {attempt}")
    if attempt == 2:
        print("Success!")
        break
    time.sleep(0.1)
else:
    # Runs only if loop exhausted without break
    print("All retries failed")

Attempt 1
Attempt 2
Success!


1.4 Functions Defining Functions

In [54]:
def build_prompt(system: str, user: str, temperature: float = 0.7) -> str:
    """Assemble a prompt string for an LLM API call.

    Args:
        system:      The system instruction.
        user:        The user message.
        temperature: Sampling temperature (0.0 - 2.0).

    Returns:
        Formatted prompt string.
    """
    return f"[System]\n{system}\n\n[User]\n{user}"

result = build_prompt(
    system="You are a concise AI assistant.",
    user="What is backpropagation?",
)
print(result)

[System]
You are a concise AI assistant.

[User]
What is backpropagation?


*args dan **kwargs

In [55]:
# *args - variable positional arguments
def log_messages(*messages: str) -> None:
    for msg in messages:
        print(f"[LOG] {msg}")

log_messages("Starting", "Loading model", "Done")

# **kwargs - variable keyword arguments
def create_api_payload(model: str, **kwargs) -> dict:
    payload = {"model": model}
    payload.update(kwargs)
    return payload

payload = create_api_payload(
    "claude-sonnet-4-5",
    max_tokens=1024,
    temperature=0.3,
    stream=True,
)
print(payload)
# {'model': 'claude-sonnet-4-5', 'max_tokens': 1024, 'temperature': 0.3, 'stream': True}

[LOG] Starting
[LOG] Loading model
[LOG] Done
{'model': 'claude-sonnet-4-5', 'max_tokens': 1024, 'temperature': 0.3, 'stream': True}


Lambda Functions

In [56]:
responses = [
    {"model": "gpt-4o",              "tokens": 540},
    {"model": "claude-sonnet-4-5",   "tokens": 310},
    {"model": "gemini-1.5-pro",      "tokens": 820},
]

# Sort by token count ascending
sorted_responses = sorted(responses, key=lambda r: r["tokens"])
for r in sorted_responses:
    print(f"{r['model']}: {r['tokens']} tokens")

claude-sonnet-4-5: 310 tokens
gpt-4o: 540 tokens
gemini-1.5-pro: 820 tokens


1.5 Scope and Closures

In [57]:
API_KEY = "sk-test-xxx"  # module-level (global)

def get_client():
    base_url = "https://api.anthropic.com"  # local
    return f"Client({base_url}, key={API_KEY[:6]}...)"

print(get_client())

# Closure - a function that remembers its enclosing scope
def make_counter(start: int = 0):
    count = [start]  # mutable container so inner function can mutate it
    def increment():
        count[0] += 1
        return count[0]
    return increment

token_counter = make_counter()
print(token_counter())  # 1
print(token_counter())  # 2
print(token_counter())  # 3

Client(https://api.anthropic.com, key=sk-tes...)
1
2
3


1.6 Exception Handling

In [58]:
import time
from typing import Optional

class APIError(Exception):
    """Raised when an AI API returns an error response."""
    def __init__(self, message: str, status_code: int):
        super().__init__(message)
        self.status_code = status_code

def call_api_with_retry(
    prompt: str,
    max_retries: int = 3,
    backoff_seconds: float = 2.0,
) -> Optional[str]:
    """Call a mock API with exponential backoff on failure."""
    for attempt in range(1, max_retries + 1):
        try:
            # Simulate API call - replace with real client call
            if attempt < 3:
                raise APIError("Rate limit exceeded", 429)
            return f"Response to: {prompt}"

        except APIError as e:
            if e.status_code == 429 and attempt < max_retries:
                wait = backoff_seconds ** attempt
                print(f"Rate limited. Retrying in {wait:.1f}s...")
                time.sleep(0.01)  # shortened for demo
            else:
                raise

        except Exception as e:
            print(f"Unexpected error: {e}")
            raise

result = call_api_with_retry("What is a neural network?")
print(result)

Rate limited. Retrying in 2.0s...
Rate limited. Retrying in 4.0s...
Response to: What is a neural network?


1.7 Module 01 Exercises
1. Write a function token_cost(tokens: int, model: str) -> float that returns the
estimated cost using a dict of costs per 1K tokens inside the function. Raise a
ValueError if the model is unknown.
2. Build a retry decorator (using functools.wraps ) that retries a function up to n
times on any exception, with a fixed sleep between attempts. Test it with a
function that fails the first two times.
3. Write a temperature_label(t: float) -> str function that maps 0.0–0.3 to "precise" ,
0.3–0.7 to "balanced" , 0.7–1.0 to "creative" , and raises ValueError outside 0.0–
1.0.
4. Parse the string "128000 tokens, 0.005 USD per 1K" and extract the token count as
an int and the cost as a float using only string methods (no regex).

In [59]:
#Latihan 1: Fungsi Estimasi Biaya Token#

def token_cost(tokens: int, model: str) -> float:
    # Dictionary simulasi harga USD per 1000 token
    costs_per_1k = {
        "gpt-4o": 0.005,
        "claude-sonnet-4-5": 0.003,
        "gemini-1.5-pro": 0.002
    }

    # Mengecek apakah model ada di dalam dictionary
    if model not in costs_per_1k:
        raise ValueError(f"Model '{model}' is unknown.")

    # Menghitung total biaya
    return (tokens / 1000) * costs_per_1k[model]

# Tes fungsinya
print(f"Biaya 2500 token untuk GPT-4o: ${token_cost(2500, 'gpt-4o')}")

Biaya 2500 token untuk GPT-4o: $0.0125


In [60]:
#Latihan 2: Decorator Retry untuk Fungsi API#

import functools
import time

def retry(n: int, sleep_time: float):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Melakukan perulangan hingga n kali
            for attempt in range(1, n + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == n:
                        print(f"Percobaan ke-{attempt} gagal. Berhenti mencoba.")
                        raise e # Lempar error asli jika sudah batas maksimal

                    print(f"Percobaan ke-{attempt} gagal ({e}). Retrying dalam {sleep_time} detik...")
                    time.sleep(sleep_time)
        return wrapper
    return decorator

# Fungsi simulasi yang sengaja akan gagal di 2 panggilan pertama
@retry(n=3, sleep_time=1.0)
def flaky_api_call():
    if not hasattr(flaky_api_call, "calls"):
        flaky_api_call.calls = 0
    flaky_api_call.calls += 1

    if flaky_api_call.calls <= 2:
        raise ConnectionError("Jaringan tidak stabil")
    return "Berhasil memanggil API di percobaan terakhir!"

# Tes fungsinya
print(flaky_api_call())

Percobaan ke-1 gagal (Jaringan tidak stabil). Retrying dalam 1.0 detik...
Percobaan ke-2 gagal (Jaringan tidak stabil). Retrying dalam 1.0 detik...
Berhasil memanggil API di percobaan terakhir!


In [61]:
#Latihan 3: Pengkategorian Suhu (Temperature) Model#

def temperature_label(t: float) -> str:
    # Memastikan suhu ada di rentang yang valid (0.0 sampai 1.0)
    if not (0.0 <= t <= 1.0):
        raise ValueError("Temperature must be between 0.0 and 1.0")

    if t <= 0.3:
        return "precise"
    elif t <= 0.7:
        return "balanced"
    else:
        return "creative"

# Tes fungsinya
print(f"Karakter suhu 0.2: {temperature_label(0.2)}")
print(f"Karakter suhu 0.5: {temperature_label(0.5)}")
print(f"Karakter suhu 0.9: {temperature_label(0.9)}")

# Kamu bisa hapus tanda '#' di bawah ini untuk melihat error jika suhu 1.5
# print(temperature_label(1.5))

Karakter suhu 0.2: precise
Karakter suhu 0.5: balanced
Karakter suhu 0.9: creative


In [62]:
#Latihan 4: Parsing String Data (Tanpa Regex)#

text = "128000 tokens, 0.005 USD per 1K"

# 1. Pisahkan string menjadi dua bagian berdasarkan tanda koma dan spasi
parts = text.split(", ")

# 2. Ambil bagian pertama ("128000 tokens"), lalu pisah spasi dan ambil kata pertama
tokens_str = parts[0].split(" ")[0]

# 3. Ambil bagian kedua ("0.005 USD per 1K"), lalu pisah spasi dan ambil angka pertama
cost_str = parts[1].split(" ")[0]

# 4. Konversi tipe datanya
token_count = int(tokens_str)
cost = float(cost_str)

# Tes hasilnya
print(f"Jumlah token : {token_count} (Tipe data: {type(token_count)})")
print(f"Harga        : {cost} (Tipe data: {type(cost)})")

Jumlah token : 128000 (Tipe data: <class 'int'>)
Harga        : 0.005 (Tipe data: <class 'float'>)


2.1 Lists

In [63]:
# Building a conversation history
conversation = []

conversation.append({"role": "user",      "content": "Hello"})
conversation.append({"role": "assistant", "content": "Hi! How can I help?"})
conversation.append({"role": "user",      "content": "Explain RAG."})

print(len(conversation))      # 3
print(conversation[0])        # first message
print(conversation[-1])       # last message
print(conversation[1:3])      # slice: messages 1 and 2

# Useful list methods
scores = [0.91, 0.76, 0.88, 0.65, 0.95]
scores.sort(reverse=True)             # in-place sort
print(scores)                         # [0.95, 0.91, 0.88, 0.76, 0.65]
print(max(scores), min(scores))

# Remove by value vs by index
scores.remove(0.76)           # remove first occurrence of this value
popped = scores.pop()         # remove and return the last element
print(popped, scores)

3
{'role': 'user', 'content': 'Hello'}
{'role': 'user', 'content': 'Explain RAG.'}
[{'role': 'assistant', 'content': 'Hi! How can I help?'}, {'role': 'user', 'content': 'Explain RAG.'}]
[0.95, 0.91, 0.88, 0.76, 0.65]
0.95 0.65
0.65 [0.95, 0.91, 0.88]


2.2 Dictionaries

In [64]:
# Building an LLM API payload
payload = {
    "model":        "claude-sonnet-4-5",
    "max_tokens":   1024,
    "temperature":  0.7,
    "messages": [
        {"role": "user", "content": "What is attention in transformers?"}
    ],
}

# Access
print(payload["model"])               # claude-sonnet-4-5
print(payload.get("top_p", 1.0))      # default 1.0 if key missing

# Update
payload["temperature"] = 0.3
payload.update({"stream": True, "top_k": 40})

# Iteration
for key, value in payload.items():
    if not isinstance(value, list):
        print(f"{key}: {value}")

# Membership check
print("stream" in payload)   # True
print("top_p" in payload)    # False

claude-sonnet-4-5
1.0
model: claude-sonnet-4-5
max_tokens: 1024
temperature: 0.3
stream: True
top_k: 40
True
False


**Nested Dicts and Default Values**

In [65]:
from collections import defaultdict

# Track token usage per model
usage: dict[str, dict[str, int]] = defaultdict(lambda: {"input": 0, "output": 0})

usage["claude-sonnet-4-5"]["input"] += 350
usage["claude-sonnet-4-5"]["output"] += 210
usage["gpt-4o"]["input"] += 420
usage["gpt-4o"]["output"] += 180

for model, counts in usage.items():
    total = counts["input"] + counts["output"]
    print(f"{model}: {total} total tokens")

claude-sonnet-4-5: 560 total tokens
gpt-4o: 600 total tokens


2.3 Sets

In [66]:
# Deduplication
retrieved_doc_ids = ["doc_3", "doc_1", "doc_3", "doc_7", "doc_1"]
unique_ids = set(retrieved_doc_ids)
print(unique_ids)  # {'doc_1', 'doc_3', 'doc_7'}

# Set operations - useful for keyword and topic analysis
gpt4_topics   = {"coding", "math", "reasoning", "vision"}
claude_topics = {"coding", "writing", "reasoning", "safety"}

both      = gpt4_topics & claude_topics  # intersection
either    = gpt4_topics | claude_topics  # union
gpt_only  = gpt4_topics - claude_topics  # difference

print("Both:     ", both)
print("Either:   ", either)
print("GPT only: ", gpt_only)

{'doc_1', 'doc_7', 'doc_3'}
Both:      {'reasoning', 'coding'}
Either:    {'reasoning', 'coding', 'writing', 'math', 'safety', 'vision'}
GPT only:  {'vision', 'math'}


**2.4 Tuples**

In [67]:
# Return multiple values from a function
def parse_model_string(model_id: str) -> tuple[str, str, str]:
    """Parse 'provider/model-name:version' into parts."""
    provider, rest = model_id.split("/")
    if ":" in rest:
        name, version = rest.split(":")
    else:
        name, version = rest, "latest"
    return provider, name, version

provider, name, version = parse_model_string("anthropic/claude-sonnet-4-5:20241022")
print(f"Provider:{provider}, Model:{name}, Version:{version}")

# Named tuples - self-documenting tuples
from typing import NamedTuple

class EmbeddingResult(NamedTuple):
    text:   str
    vector: list[float]
    model:  str

result = EmbeddingResult(
    text="Hello world",
    vector=[0.12, -0.34, 0.89],
    model="text-embedding-3-small",
)
print(result.text, result.model)

Provider:anthropic, Model:claude-sonnet-4-5, Version:20241022
Hello world text-embedding-3-small


2.5 Comprehensions (List)

In [68]:
messages = [
    {"role": "user",      "content": "What is RAG?"},
    {"role": "assistant", "content": "RAG stands for Retrieval-Augmented Generation."},
    {"role": "user",      "content": "Give an example."},
]

# Extract only user messages
user_messages = [m["content"] for m in messages if m["role"] == "user"]
print(user_messages)

# Word count per message
word_counts = [len(m["content"].split()) for m in messages]
print(word_counts)  # [3, 7, 3]

# Flatten a nested list
keywords = [["RAG", "retrieval"], ["LLM", "embedding"], ["vector"]]
flat     = [kw for group in keywords for kw in group]
print(flat)

['What is RAG?', 'Give an example.']
[3, 5, 3]
['RAG', 'retrieval', 'LLM', 'embedding', 'vector']


Dict and Set Comprehensions

In [69]:
models          = ["gpt-4o", "claude-sonnet-4-5", "gemini-1.5-pro"]
context_windows = [128_000, 200_000, 1_000_000]

# Dict comprehension
model_context = {m: c for m, c in zip(models, context_windows)}
print(model_context)

# Filter to models with > 150K context
large_context = {m: c for m, c in model_context.items() if c > 150_000}
print(large_context)

# Set comprehension - unique word lengths
sentence       = "the quick brown fox jumps over the lazy dog"
unique_lengths = {len(w) for w in sentence.split()}
print(sorted(unique_lengths))   # [2, 3, 4, 5]

{'gpt-4o': 128000, 'claude-sonnet-4-5': 200000, 'gemini-1.5-pro': 1000000}
{'claude-sonnet-4-5': 200000, 'gemini-1.5-pro': 1000000}
[3, 4, 5]


2.6 Generators

In [70]:
from typing import Generator

def chunk_text(
    text:       str,
    chunk_size: int = 500,
    overlap:    int = 50,
) -> Generator[str, None, None]:
    """Yield overlapping text chunks for embedding/RAG pipelines."""
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        yield text[start:end]
        start += chunk_size - overlap

# Load a long document (simulated)
document = "Python is a versatile language. " * 30

# Process chunks without loading all into memory at once
chunk_count = 0
for chunk in chunk_text(document, chunk_size=100, overlap=20):
    chunk_count += 1
    # In real code: embed the chunk and store in vector DB

print(f"Processed {chunk_count} chunks")

# Generator expression - like a list comprehension but lazy
sizes = (len(chunk) for chunk in chunk_text(document, 100, 20))
print(f"Max chunk size: {max(sizes)}")

Processed 12 chunks
Max chunk size: 100


2.7 Module 02 Exercises
1. Given a list of API response dicts (each with "tokens" and "latency_ms" keys),
write a comprehension that returns only responses where latency is under
500ms, sorted by token count ascending.
2. Build a function conversation_stats(messages: list[dict]) -> dict that returns a dict
with keys "total_messages" , "user_turns" , "assistant_turns" , and
"avg_words_per_message" .
3. Write a generator batch_items(items, batch_size) that yields lists of batch_size
items from any iterable. Handle the last partial batch correctly.
4. Use set operations to find models that appear in both a fast_models list and a
cheap_models list - models that are both fast and cheap.

In [71]:
# --- LATIHAN 1: Filter & Sort API Responses ---
api_responses = [
    {"tokens": 150, "latency_ms": 450},
    {"tokens": 300, "latency_ms": 600},
    {"tokens": 50,  "latency_ms": 200},
    {"tokens": 500, "latency_ms": 480}
]
# Mengambil response di bawah 500ms, diurutkan berdasarkan jumlah token
fast_responses = sorted([r for r in api_responses if r["latency_ms"] < 500], key=lambda x: x["tokens"])
print("Hasil Latihan 1:", fast_responses)


# --- LATIHAN 2: Fungsi Statistik Percakapan ---
def conversation_stats(messages: list[dict]) -> dict:
    total = len(messages)
    user_turns = sum(1 for m in messages if m.get("role") == "user")
    assistant_turns = sum(1 for m in messages if m.get("role") == "assistant")

    total_words = sum(len(m.get("content", "").split()) for m in messages)
    avg_words = total_words / total if total > 0 else 0

    return {
        "total_messages": total,
        "user_turns": user_turns,
        "assistant_turns": assistant_turns,
        "avg_words_per_message": avg_words
    }

msgs = [
    {"role": "user", "content": "Tolong jelaskan RAG"},
    {"role": "assistant", "content": "RAG adalah Retrieval-Augmented Generation"}
]
print("\nHasil Latihan 2:", conversation_stats(msgs))


# --- LATIHAN 3: Generator Batch Items ---
def batch_items(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

data_dokumen = [1, 2, 3, 4, 5, 6, 7, 8]
print("\nHasil Latihan 3:", list(batch_items(data_dokumen, 3)))


# --- LATIHAN 4: Set Operations untuk Model AI ---
fast_models  = ["gpt-4o-mini", "gemini-1.5-flash", "claude-haiku"]
cheap_models = ["claude-haiku", "gpt-4o-mini", "llama-3-8b"]

# Mencari irisan (model yang cepat DAN murah)
fast_and_cheap = set(fast_models) & set(cheap_models)
print("\nHasil Latihan 4:", fast_and_cheap)

Hasil Latihan 1: [{'tokens': 50, 'latency_ms': 200}, {'tokens': 150, 'latency_ms': 450}, {'tokens': 500, 'latency_ms': 480}]

Hasil Latihan 2: {'total_messages': 2, 'user_turns': 1, 'assistant_turns': 1, 'avg_words_per_message': 3.5}

Hasil Latihan 3: [[1, 2, 3], [4, 5, 6], [7, 8]]

Hasil Latihan 4: {'claude-haiku', 'gpt-4o-mini'}


3.1 Classes and Objects

In [72]:
from typing import Optional

class ConversationHistory:
    """Manages a rolling window of messages for LLM context."""

    def __init__(self, max_turns: int = 10, system_prompt: str = ""):
        self.system_prompt = system_prompt
        self.max_turns = max_turns
        self._messages: list[dict] = []

    def add(self, role: str, content: str) -> None:
        """Add a message and trim to max_turns."""
        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role: {role!r}")
        self._messages.append({"role": role, "content": content})

        if len(self._messages) > self.max_turns * 2:
            self._messages = self._messages[-(self.max_turns * 2):]

    def to_api_payload(self) -> list[dict]:
        """Return messages formatted for an LLM API call."""
        return list(self._messages)

    def clear(self) -> None:
        self._messages = []

    def __len__(self) -> int:
        return len(self._messages)

    def __repr__(self) -> str:
        return f"ConversationHistory(turns={len(self._messages)}, max={self.max_turns})"

# Usage
history = ConversationHistory(max_turns=5, system_prompt="Be concise.")
history.add("user", "What is an embedding?")
history.add("assistant", "An embedding is a vector representation of data.")
history.add("user", "Give a use case.")
print(history)         # ConversationHistory(turns=3, max=5)
print(len(history))    # 3

ConversationHistory(turns=3, max=5)
3


3.2 Inheritance

In [73]:
from abc import ABC, abstractmethod

class BaseLLMClient(ABC):
    """Abstract base class for LLM provider clients."""

    def __init__(self, api_key: str, model: str):
        self.api_key = api_key
        self.model = model

    @abstractmethod
    def complete(self, messages: list[dict], **kwargs) -> str:
        """Send messages and return the assistant's reply."""

    def count_words(self, text: str) -> int:
        """Shared utility - word count estimate."""
        return len(text.split())

class MockAnthropicClient(BaseLLMClient):
    """Simulates an Anthropic API client for testing."""

    def complete(self, messages: list[dict], **kwargs) -> str:
        last_user = next(
            m["content"] for m in reversed(messages) if m["role"] == "user"
        )
        return f"[Mock Anthropic] Echo: {last_user}"

class MockOpenAIClient(BaseLLMClient):
    """Simulates an OpenAI API client for testing."""

    def complete(self, messages: list[dict], **kwargs) -> str:
        return f"[Mock OpenAI] Received {len(messages)} messages."

# Polymorphism - same interface, different implementation
clients: list[BaseLLMClient] = [
    MockAnthropicClient("key-ant", "claude-sonnet-4-5"),
    MockOpenAIClient("key-oai", "gpt-4o"),
]

msgs = [{"role": "user", "content": "Hello!"}]
for client in clients:
    print(client.complete(msgs))

[Mock Anthropic] Echo: Hello!
[Mock OpenAI] Received 1 messages.


3.3 Decorators

In [74]:
import functools
import time

def log_call(func):
    """Decorator: log function name and return value."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f">>> Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"<<< {func.__name__} returned: {result!r}")
        return result
    return wrapper

def cache_result(func):
    """Simple in-memory cache (no expiry)."""
    _cache: dict = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in _cache:
            _cache[args] = func(*args)
        return _cache[args]
    return wrapper

@log_call
@cache_result
def get_embedding(text: str) -> list[float]:
    """Simulate an embedding API call (cached)."""
    time.sleep(0.01)  # simulate latency
    return [hash(text) % 100 / 100.0, 0.42, 0.87]

# First call: logs + computes
e1 = get_embedding("What is RAG?")
# Second call: logs but returns from cache instantly
e2 = get_embedding("What is RAG?")
print(e1 == e2)  # True

>>> Calling get_embedding
<<< get_embedding returned: [0.85, 0.42, 0.87]
>>> Calling get_embedding
<<< get_embedding returned: [0.85, 0.42, 0.87]
True


Parametrised Decorators

In [75]:
import functools
import time

def retry(max_attempts: int = 3, delay: float = 0.1):
    """Parametrised retry decorator."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_error
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.05)
def flaky_api_call(prompt: str) -> str:
    import random
    if random.random() < 0.6:  # fails 60% of the time
        raise ConnectionError("Simulated network error")
    return f"Response to: {prompt}"

# Panggil fungsi ini beberapa kali untuk melihat hasil simulasi jaringannya
print(flaky_api_call("Test RAG Pipeline"))

Response to: Test RAG Pipeline


3.4 Dataclasses

In [76]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class LLMConfig:
    """Configuration for a single LLM API call."""
    model:          str
    temperature:    float = 0.7
    max_tokens:     int   = 1024
    top_p:          float = 1.0
    stop_sequences: list[str] = field(default_factory=list)
    system_prompt:  Optional[str] = None

    def __post_init__(self):
        if not 0.0 <= self.temperature <= 2.0:
            raise ValueError(f"temperature must be 0-2, got {self.temperature}")
        if self.max_tokens < 1:
            raise ValueError("max_tokens must be >= 1")

    @property
    def as_dict(self) -> dict:
        """Return dict suitable for passing to an API client."""
        d = {
            "model":       self.model,
            "temperature": self.temperature,
            "max_tokens":  self.max_tokens,
            "top_p":       self.top_p,
        }
        if self.stop_sequences:
            d["stop_sequences"] = self.stop_sequences
        if self.system_prompt:
            d["system"] = self.system_prompt
        return d

cfg = LLMConfig(
    model="claude-sonnet-4-5",
    temperature=0.3,
    system_prompt="You are a concise Python tutor.",
)
print(cfg)
print(cfg.as_dict)

LLMConfig(model='claude-sonnet-4-5', temperature=0.3, max_tokens=1024, top_p=1.0, stop_sequences=[], system_prompt='You are a concise Python tutor.')
{'model': 'claude-sonnet-4-5', 'temperature': 0.3, 'max_tokens': 1024, 'top_p': 1.0, 'system': 'You are a concise Python tutor.'}


3.6 Module 03 Exercises
1. Implement a RateLimiter class with a method check_and_wait() that ensures no
more than N calls per minute, sleeping as needed. Test it by calling it 5 times
rapidly.
2. Create a PromptTemplate dataclass with a template string field and a
render(**kwargs) method that fills in placeholders using str.format_map . Add
validation that all placeholders are provided.

Phase 1 - Python Core for AI Page 3324

3. Write a @retry(max_attempts=3, delay=0.1) parametrised decorator - the syntax is
@retry(max_attempts=3) . Test it on a function that raises on the first two calls.
4. Organise the ConversationHistory class from section 3.1 and the LLMConfig
dataclass from section 3.4 into a package structure with proper __init__.py
exports.

In [77]:
# --- LATIHAN 1: RateLimiter Class ---
import time
from collections import deque

class RateLimiter:
    def __init__(self, max_calls: int, period: float = 60.0):
        self.max_calls = max_calls
        self.period = period
        self.calls = deque()

    def check_and_wait(self):
        now = time.time()
        # Buang catatan waktu yang sudah lewat dari batas period
        while self.calls and now - self.calls[0] > self.period:
            self.calls.popleft()

        # Jika kuota penuh, tunggu sampai ada slot kosong
        if len(self.calls) >= self.max_calls:
            wait_time = self.period - (now - self.calls[0])
            print(f"[RateLimiter] Kuota penuh. Menunggu {wait_time:.2f} detik...")
            time.sleep(wait_time)
            now = time.time()

        self.calls.append(now)
        print(f"Panggilan dieksekusi pada: {time.strftime('%X')}")

print("--- Hasil Latihan 1 ---")
# Tes: Maksimal 3 panggilan per 2 detik
limiter = RateLimiter(max_calls=3, period=2.0)
for i in range(5):
    limiter.check_and_wait()


# --- LATIHAN 2: PromptTemplate Dataclass ---
from dataclasses import dataclass
import string

@dataclass
class PromptTemplate:
    template: str

    def render(self, **kwargs) -> str:
        # Cari semua placeholder {text} di dalam template
        parsed = string.Formatter().parse(self.template)
        required_keys = {field_name for _, field_name, _, _ in parsed if field_name is not None}

        # Validasi apakah ada kunci yang lupa dimasukkan
        missing_keys = required_keys - set(kwargs.keys())
        if missing_keys:
            raise KeyError(f"Missing values for placeholders: {', '.join(missing_keys)}")

        return self.template.format_map(kwargs)

print("\n--- Hasil Latihan 2 ---")
tpl = PromptTemplate("Halo {name}, sistem mendeteksi kamu menggunakan bahasa {lang}.")
print(tpl.render(name="Annisa", lang="Python"))


# --- LATIHAN 3: Parametrised Retry Decorator ---
import functools

def retry(max_attempts: int = 3, delay: float = 0.1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    if attempt < max_attempts:
                        print(f"[Retry] Percobaan {attempt} gagal. Mencoba lagi dalam {delay} detik...")
                        time.sleep(delay)
            raise last_error
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.1)
def test_func():
    if not hasattr(test_func, "calls"):
        test_func.calls = 0
    test_func.calls += 1

    # Sengaja digagalkan 2 kali pertama
    if test_func.calls <= 2:
        raise ValueError("Gagal sengaja!")
    return "Berhasil pada percobaan ke-3!"

print("\n--- Hasil Latihan 3 ---")
print(test_func())


# --- LATIHAN 4: Package Structure ---
# Catatan: Karena Google Colab berbasis Notebook (satu layar memanjang),
# kita tidak bisa secara visual membuat struktur folder/package di sini dengan mudah.
# Jadi, tidak ada kode yang perlu di-run untuk soal nomor 4 ini.
# Cukup dipahami teorinya sesuai materi 3.5.
print("\n--- Hasil Latihan 4 ---")
print("Teori struktur direktori dipahami.")

--- Hasil Latihan 1 ---
Panggilan dieksekusi pada: 07:29:46
Panggilan dieksekusi pada: 07:29:46
Panggilan dieksekusi pada: 07:29:46
[RateLimiter] Kuota penuh. Menunggu 2.00 detik...
Panggilan dieksekusi pada: 07:29:48
Panggilan dieksekusi pada: 07:29:48

--- Hasil Latihan 2 ---
Halo Annisa, sistem mendeteksi kamu menggunakan bahasa Python.

--- Hasil Latihan 3 ---
[Retry] Percobaan 1 gagal. Mencoba lagi dalam 0.1 detik...
[Retry] Percobaan 2 gagal. Mencoba lagi dalam 0.1 detik...
Berhasil pada percobaan ke-3!

--- Hasil Latihan 4 ---
Teori struktur direktori dipahami.


4.1 Working with Files

In [78]:
import pathlib

# pathlib is preferred over os.path in Python 3.6+
data_dir = pathlib.Path("data")
data_dir.mkdir(exist_ok=True)

# Write a prompt template
template = """You are a {role}.
Answer the following question concisely.

Question: {question}"""

template_file = data_dir / "qa_prompt.txt"
template_file.write_text(template, encoding="utf-8")

# Read it back and fill placeholders
loaded = template_file.read_text(encoding="utf-8")
filled = loaded.format(role="Python tutor", question="What is a generator?")
print(filled)

# List all .txt files in a directory
for f in data_dir.glob("*.txt"):
    print(f.name, f.stat().st_size, "bytes")

You are a Python tutor.
Answer the following question concisely.

Question: What is a generator?
qa_prompt.txt 80 bytes


JSON

In [79]:
import json
import pathlib

# Save model evaluation results
results = {
    "model": "claude-sonnet-4-5",
    "benchmark": "MMLU",
    "scores": {"science": 0.91, "math": 0.88, "history": 0.85},
    "total_samples": 14042,
    "timestamp": "2025-01-15T09:30:00Z",
}

out = pathlib.Path("results.json")
out.write_text(json.dumps(results, indent=2), encoding="utf-8")

# Read back and use
data = json.loads(out.read_text(encoding="utf-8"))
print(f"Model: {data['model']}")
avg = sum(data["scores"].values()) / len(data["scores"])
print(f"Average score: {avg:.2f}")

Model: claude-sonnet-4-5
Average score: 0.88


CSV

In [80]:
import csv
import pathlib
from collections import defaultdict

# Write evaluation results as CSV
rows = [
    {"prompt": "What is RAG?", "model": "claude-sonnet-4-5", "tokens": 312, "latency_ms": 410},
    {"prompt": "Explain embeddings", "model": "claude-sonnet-4-5", "tokens": 498, "latency_ms": 580},
    {"prompt": "What is RAG?", "model": "gpt-4o", "tokens": 287, "latency_ms": 360},
]

csv_file = pathlib.Path("eval.csv")
with csv_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["prompt", "model", "tokens", "latency_ms"])
    writer.writeheader()
    writer.writerows(rows)

# Read and aggregate by model
model_latencies: dict[str, list[float]] = defaultdict(list)

with csv_file.open(encoding="utf-8") as f:
    for row in csv.DictReader(f):
        model_latencies[row["model"]].append(float(row["latency_ms"]))

for model, latencies in model_latencies.items():
    avg = sum(latencies) / len(latencies)
    print(f"{model}: avg latency {avg:.0f}ms")

claude-sonnet-4-5: avg latency 495ms
gpt-4o: avg latency 360ms


4.2 Async/Await

In [81]:
import asyncio

async def call_model_mock(model: str, prompt: str) -> dict:
    """Simulate calling an LLM API."""
    await asyncio.sleep(0.05)  # simulate network latency
    return {
        "model": model,
        "response": f"[{model}] Answer to: {prompt[:30]}",
        "tokens": 42,
    }

async def compare_models(prompt: str, models: list[str]) -> list[dict]:
    """Call multiple models in parallel and return all responses."""
    tasks = [call_model_mock(m, prompt) for m in models]
    results = await asyncio.gather(*tasks)
    return list(results)

# --- Perubahan Khusus untuk Google Colab (Jupyter) ---
# Alih-alih asyncio.run(...), kita langsung panggil 'await'
responses = await compare_models(
    "What is RAG?",
    ["claude-sonnet-4-5", "gpt-4o", "gemini-1.5-pro"]
)

for r in responses:
    print(f"{r['model']}: {r['response']}")

claude-sonnet-4-5: [claude-sonnet-4-5] Answer to: What is RAG?
gpt-4o: [gpt-4o] Answer to: What is RAG?
gemini-1.5-pro: [gemini-1.5-pro] Answer to: What is RAG?


Simulating LLM API Calls

In [82]:
import asyncio

async def call_model_mock(model: str, prompt: str) -> dict:
    """Simulate calling an LLM API (replace with real client call)."""
    await asyncio.sleep(0.05)  # simulate network latency
    return {
        "model": model,
        "response": f"[{model}] Answer to: {prompt[:30]}",
        "tokens": 42,
    }

async def compare_models(prompt: str, models: list[str]) -> list[dict]:
    """Call multiple models in parallel and return all responses."""
    tasks = [call_model_mock(m, prompt) for m in models]
    results = await asyncio.gather(*tasks)
    return list(results)

# --- CATATAN PENTING UNTUK GOOGLE COLAB ---
# Di gambar modul, kodenya menggunakan: responses = asyncio.run(...)
# Tapi di Google Colab, cara itu akan bikin error. Kamu harus pakai 'await' langsung seperti ini:

responses = await compare_models(
    "What is RAG?",
    ["claude-sonnet-4-5", "gpt-4o", "gemini-1.5-pro"]
)

for r in responses:
    print(f"{r['model']}: {r['response']}")

claude-sonnet-4-5: [claude-sonnet-4-5] Answer to: What is RAG?
gpt-4o: [gpt-4o] Answer to: What is RAG?
gemini-1.5-pro: [gemini-1.5-pro] Answer to: What is RAG?


4.3 Environment Variables and Secrets

In [83]:
import os
# Pastikan library python-dotenv sudah terinstall (biasanya sudah ada, atau jalankan !pip install python-dotenv)
from dotenv import load_dotenv

# Memuat file .env ke dalam sistem (jika ada)
load_dotenv()

def get_api_key(provider: str) -> str:
    """Retrieve an API key from the environment."""
    key_map = {
        "anthropic": "ANTHROPIC_API_KEY",
        "openai":    "OPENAI_API_KEY",
        "google":    "GOOGLE_API_KEY",
    }

    env_var = key_map.get(provider.lower())
    if not env_var:
        raise ValueError(f"Unknown provider: {provider}")

    key = os.getenv(env_var)
    if not key:
        raise EnvironmentError(
            f"{env_var} is not set. Add it to your .env file."
        )
    return key

# Contoh penggunaan (ini akan memicu error jika variabel environment belum diset)
try:
    anthropic_key = get_api_key("anthropic")
except Exception as e:
    print(f"Catatan Keamanan: {e}")

Catatan Keamanan: ANTHROPIC_API_KEY is not set. Add it to your .env file.


4.4 Module 04 Exercises
1. Write save_conversation(history: list[dict], path: str) and load_conversation(path:
str) -> list[dict] that serialise and deserialise conversation history to/from a
JSON file.
2. Build an async function compare_endpoints(urls: list[str]) that fetches all URLs
concurrently with httpx and returns a list of (url, status_code, response_time_ms)
tuples.
3. Create a config loader that reads a JSON config file and merges it with
environment variable overrides (env var values take precedence). Use pathlib
for file operations.
4. Write a CSV log writer class that appends a row each time an LLM is called,
recording: timestamp, model, input_tokens, output_tokens, latency_ms. Make it
thread-safe with a threading.Lock .

In [84]:
# --- LATIHAN 1: Simpan dan Muat Riwayat Percakapan (JSON) ---
import json
import pathlib

def save_conversation(history: list[dict], path: str) -> None:
    pathlib.Path(path).write_text(json.dumps(history, indent=2), encoding="utf-8")

def load_conversation(path: str) -> list[dict]:
    return json.loads(pathlib.Path(path).read_text(encoding="utf-8"))

print("--- Hasil Latihan 1 ---")
sample_history = [{"role": "user", "content": "Halo AI!"}, {"role": "assistant", "content": "Halo! Ada yang bisa dibantu?"}]
save_conversation(sample_history, "chat_history.json")
loaded_history = load_conversation("chat_history.json")
print("Berhasil dimuat dari file:", loaded_history)


# --- LATIHAN 2: Async Endpoints Fetcher dengan Httpx ---
import asyncio
import httpx

async def compare_endpoints(urls: list[str]) -> list[tuple[str, int, float]]:
    """Mengambil beberapa URL secara konkuren dan mengembalikan (url, status_code, response_time_ms)"""
    async def fetch_one(client, url):
        start_time = asyncio.get_event_loop().time()
        try:
            response = await client.get(url, timeout=5.0)
            elapsed = (asyncio.get_event_loop().time() - start_time) * 1000
            return (url, response.status_code, elapsed)
        except Exception:
            elapsed = (asyncio.get_event_loop().time() - start_time) * 1000
            return (url, 0, elapsed) # 0 menandakan gagal/timeout

    async with httpx.AsyncClient() as client:
        tasks = [fetch_one(client, url) for url in urls]
        return await asyncio.gather(*tasks)

print("\n--- Hasil Latihan 2 ---")
test_urls = [
    "https://httpbin.org/delay/1",
    "https://httpbin.org/status/200",
    "https://httpbin.org/json"
]
# Jalankan di Colab menggunakan await
results_latency = await compare_endpoints(test_urls)
for res in results_latency:
    print(f"URL: {res[0]} | Status: {res[1]} | Waktu: {res[2]:.2f} ms")


# --- LATIHAN 3: Config Loader dengan Env Overrides ---
def load_config_with_env(json_path: str) -> dict:
    path = pathlib.Path(json_path)
    # 1. Baca dari file JSON jika ada, jika tidak buat default
    if path.exists():
        config = json.loads(path.read_text(encoding="utf-8"))
    else:
        config = {"model": "gpt-4o", "temperature": 0.7}

    # 2. Timpa dengan Environment Variable jika tersedia (Env mengambil prioritas utama)
    if os.getenv("AI_MODEL"):
        config["model"] = os.getenv("AI_MODEL")
    if os.getenv("AI_TEMPERATURE"):
        config["temperature"] = float(os.getenv("AI_TEMPERATURE"))

    return config

print("\n--- Hasil Latihan 3 ---")
pathlib.Path("config.json").write_text(json.dumps({"model": "claude-3-opus", "temperature": 0.5}), encoding="utf-8")
print("Config awal:", load_config_with_env("config.json"))


# --- LATIHAN 4: Thread-Safe CSV Log Writer ---
import threading
from datetime import datetime

class CSVLogWriter:
    def __init__(self, filepath: str):
        self.filepath = pathlib.Path(filepath)
        self.lock = threading.Lock()

        # Buat header jika file belum ada
        if not self.filepath.exists():
            with self.filepath.open("w", encoding="utf-8") as f:
                f.write("timestamp,model,input_tokens,output_tokens,latency_ms\n")

    def log(self, model: str, input_tokens: int, output_tokens: int, latency_ms: float):
        # Menggunakan thread lock agar aman saat diakses banyak thread sekaligus
        with self.lock:
            timestamp = datetime.utcnow().isoformat()
            line = f"{timestamp},{model},{input_tokens},{output_tokens},{latency_ms}\n"
            with self.filepath.open("a", encoding="utf-8") as f:
                f.write(line)

print("\n--- Hasil Latihan 4 ---")
logger = CSVLogWriter("llm_audit.csv")
logger.log("gpt-4o", 120, 45, 320.4)
logger.log("claude-sonnet-4-5", 200, 80, 410.1)
print("Log berhasil ditulis dengan aman ke 'llm_audit.csv'.")
print("Isi file log:", pathlib.Path("llm_audit.csv").read_text(encoding="utf-8"))

--- Hasil Latihan 1 ---
Berhasil dimuat dari file: [{'role': 'user', 'content': 'Halo AI!'}, {'role': 'assistant', 'content': 'Halo! Ada yang bisa dibantu?'}]

--- Hasil Latihan 2 ---
URL: https://httpbin.org/delay/1 | Status: 200 | Waktu: 1150.93 ms
URL: https://httpbin.org/status/200 | Status: 200 | Waktu: 171.81 ms
URL: https://httpbin.org/json | Status: 200 | Waktu: 831.39 ms

--- Hasil Latihan 3 ---
Config awal: {'model': 'claude-3-opus', 'temperature': 0.5}

--- Hasil Latihan 4 ---
Log berhasil ditulis dengan aman ke 'llm_audit.csv'.
Isi file log: timestamp,model,input_tokens,output_tokens,latency_ms
2026-09-23T06:19:09.513405,gpt-4o,120,45,320.4
2026-09-23T06:19:09.513784,claude-sonnet-4-5,200,80,410.1
2026-09-23T07:29:50.128495,gpt-4o,120,45,320.4
2026-09-23T07:29:50.128760,claude-sonnet-4-5,200,80,410.1



/tmp/ipykernel_1654/2281709340.py:89: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()


5.1 NumPy (Arrays and Dtypes)

In [85]:
import numpy as np

# Creating arrays
scores = np.array([0.91, 0.76, 0.88, 0.65, 0.95], dtype=np.float32)
print(scores.dtype, scores.shape)  # float32 (5,)

# Zeros, ones, ranges
zeros = np.zeros((3, 4))             # 3 rows, 4 cols of 0.0
rng_vals = np.arange(0, 1.0, 0.1)    # [0.0, 0.1, ..., 0.9]
linspace = np.linspace(0, 1, 5)       # [0.0, 0.25, 0.5, 0.75, 1.0]

# Random - use a seeded Generator for reproducibility
rng = np.random.default_rng(seed=42)
mock_embedding = rng.standard_normal(1536)  # 1536-dim like text-embedding-3-small
print(f"Embedding shape: {mock_embedding.shape}, mean: {mock_embedding.mean():.4f}")

float32 (5,)
Embedding shape: (1536,), mean: -0.0236


Shape, Reshape, Indexing

In [86]:
import numpy as np

# Simulate 4 document embeddings of dimension 8
rng = np.random.default_rng(42)
embeddings = rng.standard_normal((4, 8))
print("Shape:", embeddings.shape)                    # (4, 8)
print("First embedding:", embeddings[0])
print("First 3 dims of all docs:\n", embeddings[:, :3])

# Reshape
flat = embeddings.flatten()      # (32,)
back = flat.reshape(4, 8)        # (4, 8)

# Boolean indexing
similarity_scores = np.array([0.91, 0.43, 0.78, 0.55])
above_threshold = embeddings[similarity_scores > 0.7]

print(f"Docs above 0.7 similarity: {above_threshold.shape[0]}")

Shape: (4, 8)
First embedding: [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519 -1.30217951
  0.1278404  -0.31624259]
First 3 dims of all docs:
 [[ 0.30471708 -1.03998411  0.7504512 ]
 [-0.01680116 -0.85304393  0.87939797]
 [ 0.36875078 -0.9588826   0.8784503 ]
 [-0.42832782 -0.35213355  0.53230919]]
Docs above 0.7 similarity: 2


Cosine Similarity - The Core of Semantic Search

In [87]:
import numpy as np

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))

def top_k_similar(
    query: np.ndarray,
    corpus: np.ndarray,
    k: int = 3,
) -> list[tuple[int, float]]:
    """Return indices and scores of the k most similar vectors."""
    # Normalise corpus rows
    norms = np.linalg.norm(corpus, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    normed = corpus / norms

    # Normalise query
    q_norm = np.linalg.norm(query)
    q_normed = query / (q_norm if q_norm > 0 else 1)

    # Matrix-vector multiply -> cosine similarities
    sims = normed @ q_normed
    top_idx = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in top_idx]

# Test pengujian
rng = np.random.default_rng(42)
corpus = rng.standard_normal((10, 8))
query = rng.standard_normal(8)

results = top_k_similar(query, corpus, k=3)
for idx, score in results:
    print(f"Doc{idx}: similarity = {score:.4f}")

Doc6: similarity = 0.5257
Doc0: similarity = 0.3804
Doc7: similarity = 0.1345


5.2 Pandas

In [88]:
import pandas as pd

data = [
    {"model": "gpt-4o",            "provider": "OpenAI",    "context_k": 128,  "cost_input": 2.50},
    {"model": "claude-sonnet-4-5", "provider": "Anthropic", "context_k": 200,  "cost_input": 3.00},
    {"model": "gemini-1.5-pro",    "provider": "Google",    "context_k": 1000, "cost_input": 1.25},
    {"model": "llama-3.1-70b",     "provider": "Meta",      "context_k": 128,  "cost_input": 0.00},
]

df = pd.DataFrame(data)
print("Shape:", df.shape)       # (4, 4)
print("\nDtypes:\n", df.dtypes)
print("\nHead:\n", df.head())
print("\nDescribe:\n", df.describe())

Shape: (4, 4)

Dtypes:
 model          object
provider       object
context_k       int64
cost_input    float64
dtype: object

Head:
                model   provider  context_k  cost_input
0             gpt-4o     OpenAI        128        2.50
1  claude-sonnet-4-5  Anthropic        200        3.00
2     gemini-1.5-pro     Google       1000        1.25
3      llama-3.1-70b       Meta        128        0.00

Describe:
          context_k  cost_input
count     4.000000    4.000000
mean    364.000000    1.687500
std     425.356321    1.344355
min     128.000000    0.000000
25%     128.000000    0.937500
50%     164.000000    1.875000
75%     400.000000    2.625000
max    1000.000000    3.000000


Selection and Filtering

In [89]:
# Select column
print("Models list:", df["model"].tolist())

# Filter rows
affordable = df[df["cost_input"] < 2.0]
print("\nAffordable models:\n", affordable)

# Multiple conditions
big_and_cheap = df[(df["context_k"] >= 128) & (df["cost_input"] < 2.0)]
print("\nBig & Cheap models:\n", big_and_cheap[["model", "context_k", "cost_input"]])

# loc (label-based) vs iloc (position-based)
print("\ndf.loc[0, 'model']:", df.loc[0, "model"])   # gpt-4o
print("df.iloc[0, 0]:", df.iloc[0, 0])             # gpt-4o

Models list: ['gpt-4o', 'claude-sonnet-4-5', 'gemini-1.5-pro', 'llama-3.1-70b']

Affordable models:
             model provider  context_k  cost_input
2  gemini-1.5-pro   Google       1000        1.25
3   llama-3.1-70b     Meta        128        0.00

Big & Cheap models:
             model  context_k  cost_input
2  gemini-1.5-pro       1000        1.25
3   llama-3.1-70b        128        0.00

df.loc[0, 'model']: gpt-4o
df.iloc[0, 0]: gpt-4o


Cleaning and Transforming Data

In [90]:
import pandas as pd
import numpy as np

# Simulasi dataset evaluasi yang kotor
raw = pd.DataFrame({
    "prompt":     ["Q1", "Q2", "Q3", "Q4", "Q5"],
    "response":   ["OK", None, "Good", "Bad", "OK"],
    "score":      [0.9, None, 0.85, 0.3, 0.88],
    "latency_ms": [410, 520, None, 390, 480],
})

# 1. Inspeksi data yang hilang
print("Data kosong per kolom:\n", raw.isnull().sum())

# 2. Mengisi nilai yang hilang (fillna dengan mean untuk score, median untuk latency)
raw["score"] = raw["score"].fillna(raw["score"].mean())
raw["latency_ms"] = raw["latency_ms"].fillna(raw["latency_ms"].median())

# 3. Menghapus baris yang kolom 'response'-nya masih kosong
clean = raw.dropna(subset=["response"]).copy()

# 4. Menambahkan kolom baru (computed column)
clean["pass"] = clean["score"] >= 0.7

print("\nData Bersih:\n", clean)
print(f"Pass rate: {clean['pass'].mean():.0%}")

Data kosong per kolom:
 prompt        0
response      1
score         1
latency_ms    1
dtype: int64

Data Bersih:
   prompt response  score  latency_ms   pass
0     Q1       OK   0.90       410.0   True
2     Q3     Good   0.85       445.0   True
3     Q4      Bad   0.30       390.0  False
4     Q5       OK   0.88       480.0   True
Pass rate: 75%


GroupBy, Aggregation, & Pivot Table

In [91]:
import pandas as pd

evals = pd.DataFrame({
    "model":      ["claude", "gpt-4o", "claude", "gpt-4o", "claude", "gpt-4o"],
    "task":       ["qa", "qa", "summarise", "summarise", "code", "code"],
    "score":      [0.91, 0.88, 0.85, 0.82, 0.93, 0.90],
    "latency_ms": [420, 380, 610, 550, 340, 300],
})

# 1. Rata-rata skor per model
print("Rata-rata skor per model:\n", evals.groupby("model")["score"].mean())

# 2. Agregasi ganda (Multiple aggregations)
summary = evals.groupby("model").agg(
    avg_score   = ("score", "mean"),
    avg_latency = ("latency_ms", "mean"),
    num_tasks   = ("task", "count"),
)
print("\nRingkasan Agregasi:\n", summary)

# 3. Pivot Table (Model vs Task)
pivot = evals.pivot_table(
    values="score", index="model", columns="task", aggfunc="mean"
)
print("\nPivot Table Model vs Task:\n", pivot)

Rata-rata skor per model:
 model
claude    0.896667
gpt-4o    0.866667
Name: score, dtype: float64

Ringkasan Agregasi:
         avg_score  avg_latency  num_tasks
model                                    
claude   0.896667   456.666667          3
gpt-4o   0.866667   410.000000          3

Pivot Table Model vs Task:
 task    code    qa  summarise
model                        
claude  0.93  0.91       0.85
gpt-4o  0.90  0.88       0.82


5.3 Mini Project - Model Evaluation Pipeline

In [92]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field

@dataclass
class EvalResult:
    prompt: str
    model: str
    response: str
    latency_ms: float
    score: float = field(default=0.0)

def mock_llm_call(prompt: str, model: str) -> tuple[str, float]:
    """Simulasi panggilan LLM. Mengembalikan (response, latency_ms)."""
    rng = np.random.default_rng(abs(hash(prompt + model)) % 2**31)
    latency = rng.uniform(300, 700)
    response = f"[{model}] Answer regarding {prompt[:20]} with attention and model details."
    return response, latency

def score_response(response: str, expected_keywords: list[str]) -> float:
    """Penilaian sederhana berbasis keyword (0.0 - 1.0)."""
    found = sum(1 for kw in expected_keywords if kw.lower() in response.lower())
    return found / len(expected_keywords) if expected_keywords else 0.0

# Menjalankan evaluasi
prompts = [
    ("What is a transformer?", ["attention", "model"]),
    ("Define RAG?", ["retrieval", "generation"]),
    ("What is fine-tuning?", ["training", "weights"]),
]
models = ["claude-sonnet-4-5", "gpt-4o"]

results: list[EvalResult] = []
for prompt, keywords in prompts:
    for model in models:
        response, latency = mock_llm_call(prompt, model)
        score = score_response(response, keywords)
        results.append(EvalResult(prompt, model, response, latency, score))

# Analisis dengan Pandas
df = pd.DataFrame([vars(r) for r in results])
summary = df.groupby("model").agg(
    avg_score   = ("score", "mean"),
    avg_latency = ("latency_ms", "mean"),
).round(3)

print("Hasil Evaluasi Model:\n", summary)

# Simpan ke CSV
df.to_csv("eval_results.csv", index=False)
print("\nBerhasil disimpan ke 'eval_results.csv'")

Hasil Evaluasi Model:
                    avg_score  avg_latency
model                                    
claude-sonnet-4-5      0.333      510.471
gpt-4o                 0.333      590.976

Berhasil disimpan ke 'eval_results.csv'


5.4 Module 05 Exercises
1. Load a CSV of LLM benchmark scores (create one with at least 20 rows and 4
models). Compute: mean score per model, best-performing task per model,
and a correlation between score and latency columns.

2. Write a function normalise_embeddings(matrix: np.ndarray) -> np.ndarray that L2-
normalises each row. Verify that all row norms equal 1.0 after normalisation.

3. Implement a function that reads a folder of .txt files and returns a Pandas
DataFrame with columns: filename , char_count , word_count , sentence_count . Sort
by word_count descending.
4. Build a pairwise cosine similarity matrix for a small corpus of 5 strings using
hash-based mock embeddings. Find and print the pair with the highest
similarity.

In [93]:
import numpy as np
import pandas as pd
import pathlib

print("=== LATIHAN 1: Benchmark Scores Analysis ===")
# Membuat dummy dataset benchmark
rng = np.random.default_rng(42)
df_bench = pd.DataFrame({
    "model": rng.choice(["gpt-4o", "claude", "gemini", "llama"], 25),
    "task":  rng.choice(["qa", "summarise", "coding", "translation"], 25),
    "score": rng.uniform(0.6, 1.0, 25),
    "latency_ms": rng.uniform(200, 800, 25)
})

# 1. Mean score per model
print("Mean score per model:\n", df_bench.groupby("model")["score"].mean())

# 2. Best-performing task per model
task_means = df_bench.groupby(["model", "task"])["score"].mean().reset_index()
best_tasks = task_means.loc[task_means.groupby("model")["score"].idxmax()]
print("\nBest performing task per model:\n", best_tasks)

# 3. Correlation between score and latency
print("\nKorelasi score dan latency:", df_bench["score"].corr(df_bench["latency_ms"]))


print("\n=== LATIHAN 2: L2 Normalisation ===")
def normalise_embeddings(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    return matrix / norms

test_mat = rng.standard_normal((3, 4))
normed_mat = normalise_embeddings(test_mat)
print("Panjang norm tiap baris (harus 1.0):", np.linalg.norm(normed_mat, axis=1))


print("\n=== LATIHAN 3: Text Folder Analyzer ===")
# Simulasi membaca file txt dari folder
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    d = pathlib.Path(tmpdir)
    (d / "doc1.txt").write_text("Hello world. GenAI is awesome!", encoding="utf-8")
    (d / "doc2.txt").write_text("Python for Data Science and Machine Learning tasks.", encoding="utf-8")

    def analyze_txt_folder(folder_path: str) -> pd.DataFrame:
        folder = pathlib.Path(folder_path)
        records = []
        for f in folder.glob("*.txt"):
            text = f.read_text(encoding="utf-8")
            records.append({
                "filename": f.name,
                "char_count": len(text),
                "word_count": len(text.split()),
                "sentence_count": text.count('.') + text.count('!') + text.count('?')
            })
        return pd.DataFrame(records).sort_values(by="word_count", ascending=False)

    print(analyze_txt_folder(tmpdir))


print("\n=== LATIHAN 4: Pairwise Cosine Similarity ===")
corpus = ["Hello world", "Artificial intelligence", "Large language models", "Python programming", "Cosine similarity"]
# Membuat mock embedding berbasis hash
embeddings = np.array([rng.standard_normal(16) for _ in corpus])
# Normalisasi L2
normed_emb = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
# Matriks kesamaan kosinus
sim_matrix = normed_emb @ normed_emb.T
np.fill_diagonal(sim_matrix, -1) # Abaikan kecocokan diri sendiri

i, j = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
print(f"Pasangan dengan kemiripan tertinggi: '{corpus[i]}' dan '{corpus[j]}' (Skor: {sim_matrix[i, j]:.4f})")

=== LATIHAN 1: Benchmark Scores Analysis ===
Mean score per model:
 model
claude    0.805708
gemini    0.790975
gpt-4o    0.799384
llama     0.757004
Name: score, dtype: float64

Best performing task per model:
     model    task     score
1  claude      qa  0.932904
3  gemini  coding  0.867926
7  gpt-4o      qa  0.987004
9   llama  coding  0.872998

Korelasi score dan latency: -0.05198216726337153

=== LATIHAN 2: L2 Normalisation ===
Panjang norm tiap baris (harus 1.0): [1. 1. 1.]

=== LATIHAN 3: Text Folder Analyzer ===
   filename  char_count  word_count  sentence_count
1  doc2.txt          51           8               1
0  doc1.txt          30           5               2

=== LATIHAN 4: Pairwise Cosine Similarity ===
Pasangan dengan kemiripan tertinggi: 'Large language models' dan 'Cosine similarity' (Skor: 0.4727)
